In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import time

In [14]:
X_train = pd.read_csv("../../../data/processed/exp_1/X_train_tree.csv")
X_test  = pd.read_csv("../../../data/processed/exp_1/X_test_tree.csv")
y_train = pd.read_csv("../../../data/processed/exp_1/y_train.csv").squeeze()
y_test  = pd.read_csv("../../../data/processed/exp_1/y_test.csv").squeeze()

In [15]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

start = time.time()
rf.fit(X_train, y_train)
training_time = time.time() - start

In [16]:
start = time.time()
y_pred = rf.predict(X_test)
inference_time = (time.time() - start) / len(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"RMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R²            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")
print(f"Inference time: {inference_time*1000_000:.4f}µs")

RMSE          : 8.8816
MAE           : 4.1286
R²            : 0.9118
Training time : 5.31s
Inference time: 32.1967µs


RandomForest, although considered a far superior model compared to Linear Regression gives larger error. Lets tune the hyperparams.

In [35]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [3, 5, 10, 15],
    'max_features': [0.5, 0.7, 1.0],
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
    param_distributions=param_grid,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
search.fit(X_train, y_train)
training_time = time.time() - start

print("Best params:", search.best_params_)

rf = search.best_estimator_

y_pred = rf.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R²            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best params: {'n_estimators': 50, 'min_samples_leaf': 3, 'max_features': 0.5, 'max_depth': 15}
RMSE          : 10.4369
MAE           : 5.4109
R²            : 0.8782
Training time : 238.97s


Even worse result with randomized search. This is because the model is validated against data from the train set itself and it is tested against test set. The parameters best for validation set is not the best for test set.